In [1]:
import sys
import os
import pandas as pd

# Add project root to Python path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
from src.entsoe_client import (
    get_day_ahead_prices,
    get_generation,
    get_wind_solar_forecast,
    get_crossborder_flows
)

In [3]:
# Define time range for data extraction
START = "2020-01-01"
END = "2020-01-07"

In [4]:
# ---------------------------------------------
#  Fetch day-ahead prices
# ---------------------------------------------
prices = get_day_ahead_prices(START, END)
prices = prices.to_frame("price_eur_mwh")
prices = prices.tz_convert("Europe/Vilnius")

In [5]:
# ---------------------------------------------
#  Fetch actual generation data
# ---------------------------------------------
gen = get_generation(START, END)

# Flatten MultiIndex columns (technology, type)
gen.columns = [
    f"{tech.lower().replace(' ', '_').replace('-', '_')}_{kind.lower().replace(' ', '_')}"
    for tech, kind in gen.columns
]

# Select only required generation columns
gen = gen[[
    "solar_actual_aggregated",
    "wind_onshore_actual_aggregated"
]]

# Rename columns to clearer names
gen = gen.rename(columns={
    "solar_actual_aggregated": "gen_solar_mw",
    "wind_onshore_actual_aggregated": "gen_wind_onshore_mw"
})

In [6]:
# ---------------------------------------------
#  Fetch wind & solar forecast
# ---------------------------------------------
forecast = get_wind_solar_forecast(START, END)

forecast = forecast.rename(columns={
    "Solar": "fc_solar_mw",
    "Wind Onshore": "fc_wind_onshore_mw"
})

forecast = forecast.tz_convert("Europe/Vilnius")

In [7]:
# ---------------------------------------------
#  Create unified hourly index
# ---------------------------------------------
idx = pd.date_range(
    START,
    END,
    freq="h",
    tz="Europe/Vilnius",
    inclusive="left"
)

In [8]:
# ---------------------------------------------
# Reindex all datasets to the same timeline
# ---------------------------------------------
prices = prices.reindex(idx)
gen = gen.tz_convert("Europe/Vilnius").reindex(idx)
forecast = forecast.reindex(idx)

In [9]:
# ---------------------------------------------
#  Fetch and align cross-border flows
# ---------------------------------------------
flows = get_crossborder_flows(START, END, tz="Europe/Vilnius")
flows = flows.reindex(idx)

In [10]:
# ---------------------------------------------
#  Combine all datasets into a single DataFrame
# ---------------------------------------------
market_df = (
    prices
    .join(gen, how="left")
    .join(forecast, how="left")
    .join(flows, how="left")
)

In [11]:
# ---------------------------------------------
#  Inspect dataset
# ---------------------------------------------
market_df.info()
market_df.head()
market_df.isna().sum()

In [12]:
# ---------------------------------------------
#  Save dataset to CSV
# ---------------------------------------------
output_path = f"../data/market_{START}_{END}.csv"
market_df.to_csv(output_path)
print("Duomenys išsaugoti į:", output_path)